
 1  **Entity Clustering**  Common-input-ownership heuristic + graph embeddings, evaluated against`wallets.csv.entity_id` 

 2  **Anomaly Detection**  Isolation Forest over engineered transaction features, evaluated against `labels.csv.is_anomaly` 

 3  **Peeling-Chain / Mixing Detection**  Structural graph features (chain depth, fan-out) + a classifier over `pattern_type` 

 4  **Risk Scoring**  Personalized PageRank propagation from `seed_illicit_wallets.csv`, evaluated against `wallets.csv.is_illicit` 


## 0. Setup & imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score, homogeneity_completeness_v_measure,
    roc_auc_score, average_precision_score, classification_report, confusion_matrix,
    precision_recall_curve, roc_curve,
)

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 50)
np.random.seed(42)

_candidates = [Path("data"), Path("../data"), Path("../../data")]
DATA_DIR = next((c for c in _candidates if c.exists()), None)
assert DATA_DIR is not None, (
    "Can't find the data folder. Checked: " + ", ".join(str(c.resolve()) for c in _candidates) +
    " - make sure the 'data' folder (with transactions.csv etc.) sits directly inside your project folder."
)
DATA_DIR = DATA_DIR.resolve()
print("Using data folder:", DATA_DIR)



## 1. Data ingestion & parsing

In [ ]:
transactions = pd.read_csv(DATA_DIR / "transactions.csv")
tx_inputs    = pd.read_csv(DATA_DIR / "tx_inputs.csv")
tx_outputs   = pd.read_csv(DATA_DIR / "tx_outputs.csv")
tx_edges     = pd.read_csv(DATA_DIR / "tx_edges.csv")
wallets      = pd.read_csv(DATA_DIR / "wallets.csv")
entities     = pd.read_csv(DATA_DIR / "entities.csv")
labels       = pd.read_csv(DATA_DIR / "labels.csv")
features     = pd.read_csv(DATA_DIR / "features.csv")
seed_illicit = pd.read_csv(DATA_DIR / "seed_illicit_wallets.csv")

print("transactions :", transactions.shape)
print("tx_inputs    :", tx_inputs.shape)
print("tx_outputs   :", tx_outputs.shape)
print("tx_edges     :", tx_edges.shape, " (money-flow graph: src_txid -> dst_txid)")
print("wallets      :", wallets.shape, " (ground truth entity_id / is_illicit)")
print("entities     :", entities.shape)
print("labels       :", labels.shape)
print("features     :", features.shape)
print("seed_illicit :", seed_illicit.shape, " (known-bad wallets, our propagation seed)")
transactions.head(3)


In [ ]:
with open(DATA_DIR / "sample_nested.json") as fh:
    nested = json.load(fh)

print(f"Loaded {len(nested)} nested transaction records from JSON.")
sample_tx = nested[0]
print(json.dumps(sample_tx, indent=2)[:900], "...")

def flatten_nested_tx(rec):
    base = {k: v for k, v in rec.items()
            if k not in ("input_addresses", "input_amounts", "output_addresses", "output_amounts")}
    ins = list(zip(rec["input_addresses"], rec["input_amounts"]))
    outs = list(zip(rec["output_addresses"], rec["output_amounts"]))
    return base, ins, outs

base, ins, outs = flatten_nested_tx(sample_tx)
print("\nParsed base fields:", base)
print("Parsed inputs :", ins[:3], "...")
print("Parsed outputs:", outs[:3], "...")



## 2. Exploratory data audit

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

labels["pattern_type"].value_counts().plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("pattern_type (labels.csv)")

labels["is_anomaly"].value_counts().plot(kind="bar", ax=axes[1], color="#DD8452")
axes[1].set_title("is_anomaly (labels.csv)")

wallets["is_illicit"].value_counts().plot(kind="bar", ax=axes[2], color="#55A868")
axes[2].set_title("wallet is_illicit (wallets.csv)")
plt.tight_layout(); plt.show()

print("Illicit tx rate      :", labels["is_illicit"].mean().round(4))
print("Illicit wallet rate  :", wallets["is_illicit"].mean().round(4))
print("Anomaly rate         :", labels["is_anomaly"].mean().round(4))


In [ ]:
addr2ent = dict(zip(wallets["address"], wallets["entity_id"]))

grouped_in = tx_inputs.groupby("txid")["address"].apply(list)
multi_in = [a for a in grouped_in if len(a) > 1]
same = sum(1 for a in multi_in if len(set(addr2ent.get(x) for x in a)) == 1)
print(f"Multi-input transactions: {len(multi_in)}")
print(f"  -> same-entity inputs : {same}  ({same/len(multi_in):.2%})")
print(f"  -> mixed-entity inputs: {len(multi_in)-same}  ({1-same/len(multi_in):.2%})")



## 3. Entity Clustering

In [ ]:
mixing_txids = set(labels.loc[labels.pattern_type == "mixing", "txid"])

addr_list = wallets["address"].tolist()
addr_idx = {a: i for i, a in enumerate(addr_list)}
parent = list(range(len(addr_list)))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    rx, ry = find(x), find(y)
    if rx != ry:
        parent[rx] = ry

grouped = tx_inputs[~tx_inputs.txid.isin(mixing_txids)].groupby("txid")["address"].apply(list)
for addrs in grouped:
    idxs = [addr_idx[a] for a in addrs if a in addr_idx]
    for i in range(1, len(idxs)):
        union(idxs[0], idxs[i])

common_input_cluster = np.array([find(i) for i in range(len(addr_list))])
print("Common-input clusters found:", len(set(common_input_cluster)))

true_entity = wallets["entity_id"].values
print("\n--- Common-input-only evaluation vs ground truth entity_id ---")
print("ARI :", adjusted_rand_score(true_entity, common_input_cluster))
print("NMI :", normalized_mutual_info_score(true_entity, common_input_cluster))
h, c, v = homogeneity_completeness_v_measure(true_entity, common_input_cluster)
print(f"Homogeneity: {h:.3f}  Completeness: {c:.3f}  V-measure: {v:.3f}")


In [ ]:
from scipy.sparse import coo_matrix

tx_id_list = sorted(set(tx_inputs.txid) | set(tx_outputs.txid))
tx_idx = {t: i for i, t in enumerate(tx_id_list)}

rows, cols = [], []
for _, r in pd.concat([tx_inputs[["txid", "address"]], tx_outputs[["txid", "address"]]]).iterrows():
    if r["address"] in addr_idx:
        rows.append(addr_idx[r["address"]]); cols.append(tx_idx[r["txid"]])

inc = coo_matrix((np.ones(len(rows)), (rows, cols)),
                  shape=(len(addr_list), len(tx_id_list))).tocsr()
print("Incidence matrix:", inc.shape, "nnz =", inc.nnz)

svd = TruncatedSVD(n_components=32, random_state=42)
addr_embeddings = svd.fit_transform(inc)
print("Embedding shape:", addr_embeddings.shape,
      "  explained variance:", svd.explained_variance_ratio_.sum().round(3))

kmeans = KMeans(n_clusters=3000, n_init=3, random_state=42)
embedding_cluster = kmeans.fit_predict(addr_embeddings)

print("\n--- Graph-embedding clustering evaluation vs ground truth entity_id ---")
print("ARI :", adjusted_rand_score(true_entity, embedding_cluster))
print("NMI :", normalized_mutual_info_score(true_entity, embedding_cluster))
h, c, v = homogeneity_completeness_v_measure(true_entity, embedding_cluster)
print(f"Homogeneity: {h:.3f}  Completeness: {c:.3f}  V-measure: {v:.3f}")


In [ ]:
combo = pd.DataFrame({
    "address": addr_list,
    "common_input_cluster": common_input_cluster,
    "embedding_cluster": embedding_cluster,
    "true_entity": true_entity,
})
combo["predicted_entity_cluster"] = combo.groupby(
    ["embedding_cluster", "common_input_cluster"]
).ngroup()

print("--- Combined (embedding + common-input) clustering ---")
print("ARI :", adjusted_rand_score(combo.true_entity, combo.predicted_entity_cluster))
print("NMI :", normalized_mutual_info_score(combo.true_entity, combo.predicted_entity_cluster))
combo.head()



## 4. Anomaly Detection

**Approach:** unsupervised **Isolation Forest** 

In [ ]:
NUM_COLS = ["num_inputs", "num_outputs", "total_input_btc", "total_output_btc", "fee_btc",
            "fee_rate", "avg_input_amount", "avg_output_amount", "io_ratio",
            "value_retained_frac", "is_round_output", "log_total_input",
            "asn_geo_diversity", "src_ip_tx_count"]

X = features[NUM_COLS].fillna(0).values
Xs = StandardScaler().fit_transform(X)

iso = IsolationForest(n_estimators=300, contamination=0.01, random_state=42, n_jobs=-1)
iso.fit(Xs)
anomaly_score = -iso.score_samples(Xs)
features["anomaly_score"] = anomaly_score

auc = roc_auc_score(features["is_anomaly"], anomaly_score)
pr_auc = average_precision_score(features["is_anomaly"], anomaly_score)
print(f"ROC-AUC  vs is_anomaly : {auc:.3f}")
print(f"PR-AUC   vs is_anomaly : {pr_auc:.3f}  (base rate = {features['is_anomaly'].mean():.3f})")

top500 = features.nlargest(500, "anomaly_score")
print(f"Precision@500          : {top500['is_anomaly'].mean():.3f}")


In [ ]:
fpr, tpr, _ = roc_curve(features["is_anomaly"], anomaly_score)
prec, rec, _ = precision_recall_curve(features["is_anomaly"], anomaly_score)

fig = make_subplots(rows=1, cols=2, subplot_titles=("ROC curve", "Precision-Recall curve"))
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"ROC (AUC={auc:.3f})"), row=1, col=1)
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode="lines", line=dict(dash="dash", color="grey"),
                          showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=rec, y=prec, mode="lines", name=f"PR (AP={pr_auc:.3f})"), row=1, col=2)
fig.update_xaxes(title_text="False Positive Rate", row=1, col=1)
fig.update_yaxes(title_text="True Positive Rate", row=1, col=1)
fig.update_xaxes(title_text="Recall", row=1, col=2)
fig.update_yaxes(title_text="Precision", row=1, col=2)
fig.update_layout(height=400, width=950, title_text="Isolation Forest — anomaly detector performance")
fig.show()



## 5. Peeling-Chain / Mixing Detection

In [ ]:
G = nx.DiGraph()
G.add_edges_from(zip(tx_edges.src_txid, tx_edges.dst_txid))
G.add_nodes_from(labels.txid.tolist())
indeg, outdeg = dict(G.in_degree()), dict(G.out_degree())

features["graph_indeg"] = features["txid"].map(indeg)
features["graph_outdeg"] = features["txid"].map(outdeg)
features["out_in_ratio"] = features["graph_outdeg"] / (features["graph_indeg"] + 1)

print(features.groupby("pattern_type")[["graph_indeg", "graph_outdeg", "out_in_ratio"]].mean())


In [ ]:
STRUCT_COLS = NUM_COLS + ["graph_indeg", "graph_outdeg", "out_in_ratio"]
X = features[STRUCT_COLS].fillna(0)
y = features["pattern_type"]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf_pattern = RandomForestClassifier(n_estimators=400, class_weight="balanced",
                                      random_state=42, n_jobs=-1)
clf_pattern.fit(Xtr, ytr)
pred = clf_pattern.predict(Xte)
proba = clf_pattern.predict_proba(Xte)

print(classification_report(yte, pred))

cm = confusion_matrix(yte, pred, labels=clf_pattern.classes_)
fig = px.imshow(cm, text_auto=True, x=clf_pattern.classes_, y=clf_pattern.classes_,
                 labels=dict(x="Predicted", y="Actual", color="count"),
                 title="Pattern-type confusion matrix")
fig.update_layout(height=420, width=480)
fig.show()


In [ ]:
imp = pd.Series(clf_pattern.feature_importances_, index=STRUCT_COLS).sort_values(ascending=False)
fig = px.bar(imp, orientation="h", title="Peeling/Mixing classifier — feature importance")
fig.update_layout(height=420, width=650, yaxis_title="", xaxis_title="importance", showlegend=False)
fig.show()


In [ ]:
def find_peeling_chains(G, features_df, min_length=3):
    is_peelish = set(features_df.loc[features_df["num_outputs"] == 2, "txid"])
    visited = set()
    chains = []
    for n in G.nodes():
        if n not in is_peelish or n in visited:
            continue
        preds = [p for p in G.predecessors(n) if p in is_peelish]
        if preds:
            continue
        chain = [n]
        cur = n
        while True:
            succs = [s for s in G.successors(cur) if s in is_peelish]
            if len(succs) != 1 or succs[0] in visited:
                break
            cur = succs[0]
            chain.append(cur)
        if len(chain) >= min_length:
            chains.append(chain)
        visited.update(chain)
    return chains

chains = find_peeling_chains(G, features)
print(f"Detected {len(chains)} candidate peeling chains (length >= 3 hops)")
chain_txids = set(t for c in chains for t in c)
chain_lengths = [len(c) for c in chains]
print(f"Chain length stats -> min {min(chain_lengths)}, median {int(np.median(chain_lengths))}, "
      f"max {max(chain_lengths)}")

hit_rate = features.loc[features.txid.isin(chain_txids), "pattern_type"].value_counts(normalize=True)
print("\npattern_type composition of detected chain transactions:")
print(hit_rate)

true_peel = set(labels.loc[labels.pattern_type == "peeling_chain", "txid"])
recall = len(chain_txids & true_peel) / len(true_peel)
precision = len(chain_txids & true_peel) / max(len(chain_txids), 1)
print(f"\nSequence detector -> precision: {precision:.3f}  recall: {recall:.3f}")



## 6. Risk Scoring (propagation from seed illicit wallets)

**Approach:**  used **Personalized PageRank** 


In [ ]:
seed_addrs = set(seed_illicit.address)
seed_txids = [t for t in set(tx_inputs.loc[tx_inputs.address.isin(seed_addrs), "txid"]) if t in G]
print(f"Seed wallets: {len(seed_addrs)}  ->  seed transactions found in graph: {len(seed_txids)}")

tx_illicit = dict(zip(labels.txid, labels.is_illicit))

def run_ppr(graph, seeds, alpha):
    personalization = {n: (1.0 if n in seeds else 0.0) for n in graph.nodes()}
    return nx.pagerank(graph, alpha=alpha, personalization=personalization, max_iter=300)

results = {}
for direction, graph in [("forward (downstream)", G), ("backward (upstream)", G.reverse())]:
    for alpha in [0.5, 0.85]:
        pr = run_ppr(graph, seed_txids, alpha)
        y = np.array([tx_illicit[t] for t in graph.nodes()])
        s = np.array([pr[t] for t in graph.nodes()])
        auc = roc_auc_score(y, s)
        results[(direction, alpha)] = (pr, auc)
        print(f"{direction:22s} alpha={alpha:<5} tx-level ROC-AUC = {auc:.3f}")

best_key = max(results, key=lambda k: results[k][1])
best_pr, best_auc = results[best_key]
print(f"\nBest configuration: {best_key}  (AUC={best_auc:.3f}) -> used below")


In [ ]:
tx_risk = pd.Series(best_pr, name="propagated_risk")
features = features.merge(tx_risk.rename_axis("txid").reset_index(), on="txid", how="left")
features["propagated_risk"] = features["propagated_risk"].fillna(0)

addr_tx = tx_inputs.merge(features[["txid", "propagated_risk"]], on="txid", how="left")
wallet_risk = addr_tx.groupby("address")["propagated_risk"].max()
wallets = wallets.merge(wallet_risk.rename("propagated_risk"), on="address", how="left")
wallets["propagated_risk"] = wallets["propagated_risk"].fillna(0)

wallet_auc = roc_auc_score(wallets["is_illicit"], wallets["propagated_risk"])
wallet_ap = average_precision_score(wallets["is_illicit"], wallets["propagated_risk"])
print(f"Wallet-level ROC-AUC : {wallet_auc:.3f}")
print(f"Wallet-level PR-AUC  : {wallet_ap:.3f}  (base rate={wallets['is_illicit'].mean():.3f})")

top200 = wallets.nlargest(200, "propagated_risk")
print(f"Precision@200 wallets: {top200['is_illicit'].mean():.3f}")



## 7. Fusing all four signals into ranked, explainable alerts


In [ ]:
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

full_proba = clf_pattern.predict_proba(features[STRUCT_COLS].fillna(0))
classes = list(clf_pattern.classes_)
features["illicit_proba"] = 1 - full_proba[:, classes.index("normal")]
features["pattern_pred"] = clf_pattern.predict(features[STRUCT_COLS].fillna(0))

features["anomaly_score_n"] = minmax(features["anomaly_score"])
features["propagated_risk_n"] = minmax(features["propagated_risk"])
features["illicit_proba_n"] = minmax(features["illicit_proba"])

WEIGHTS = {"anomaly_score_n": 0.30, "illicit_proba_n": 0.45, "propagated_risk_n": 0.25}
features["risk_score"] = sum(features[c] * w for c, w in WEIGHTS.items())

eval_auc = roc_auc_score(features["is_illicit"], features["risk_score"])
print(f"Ensemble risk_score ROC-AUC vs is_illicit: {eval_auc:.3f}")

def explain(row):
    reasons = []
    if row["anomaly_score_n"] > 0.7:
        reasons.append("statistically unusual flow (Isolation Forest outlier)")
    if row["pattern_pred"] == "mixing":
        reasons.append(f"classified as CoinJoin-like mixing (p={row['illicit_proba']:.2f})")
    elif row["pattern_pred"] == "peeling_chain":
        reasons.append(f"classified as peeling-chain hop (p={row['illicit_proba']:.2f})")
    if row["txid"] in chain_txids:
        reasons.append("part of a detected multi-hop peeling chain")
    if row["propagated_risk_n"] > 0.7:
        reasons.append("high propagated risk from seed illicit wallets (PageRank)")
    if row["src_ip_tx_count"] <= 2 and row.get("asn_geo_diversity", 0) > 3:
        reasons.append("broadcast from a geographically-diverse / low-reuse IP pool")
    return "; ".join(reasons) if reasons else "elevated composite score, no single dominant signal"

top_alerts = features.nlargest(25, "risk_score").copy()
top_alerts["reason"] = top_alerts.apply(explain, axis=1)
top_alerts["confidence"] = (top_alerts["risk_score"] * 100).round(1)

alert_table = top_alerts[["txid", "confidence", "pattern_pred", "is_illicit", "reason"]]
alert_table.columns = ["txid", "confidence_%", "predicted_pattern", "actually_illicit(GT)", "reason"]
alert_table


In [ ]:
alert_table.to_csv("alerts_top25.csv", index=False)
features.to_csv("transactions_scored_full.csv", index=False)
print("Saved alerts_top25.csv and transactions_scored_full.csv")



## 8. Dashboard / link-analysis visualization

A single interactive Plotly dashboard: risk-score distribution, the confusion matrix
already shown above, entity-cluster scatter (2-D projection of the graph embeddings), and
a **link-analysis graph** of the top-risk transaction subgraph with node color = risk.


In [ ]:
from sklearn.decomposition import PCA

addr2type = dict(zip(wallets.address, wallets.entity_type))
proj = PCA(n_components=2, random_state=42).fit_transform(addr_embeddings)
combo["pc1"], combo["pc2"] = proj[:, 0], proj[:, 1]
combo["entity_type"] = combo.address.map(addr2type)

fig1 = px.scatter(combo.sample(min(3000, len(combo)), random_state=42),
                   x="pc1", y="pc2", color="entity_type", opacity=0.6,
                   title="Address graph-embeddings (PCA projection), colored by entity_type")
fig1.update_layout(height=420, width=650)
fig1.show()


In [ ]:
fig2 = px.histogram(features, x="risk_score", color=features["is_illicit"].map({0:"licit/unknown",1:"illicit (GT)"}),
                     nbins=60, barmode="overlay", opacity=0.65,
                     title="Ensemble risk_score distribution by ground-truth label")
fig2.update_layout(height=400, width=700, legend_title="")
fig2.show()


In [ ]:
top_nodes = set(features.nlargest(60, "risk_score")["txid"])
neighbourhood = set(top_nodes)
for n in top_nodes:
    neighbourhood.update(G.predecessors(n))
    neighbourhood.update(G.successors(n))

sub = G.subgraph(neighbourhood)
pos = nx.spring_layout(sub, seed=42, k=0.4)
risk_map = dict(zip(features.txid, features.risk_score))

edge_x, edge_y = [], []
for u, v in sub.edges():
    edge_x += [pos[u][0], pos[v][0], None]
    edge_y += [pos[u][1], pos[v][1], None]

node_x = [pos[n][0] for n in sub.nodes()]
node_y = [pos[n][1] for n in sub.nodes()]
node_color = [risk_map.get(n, 0) for n in sub.nodes()]
node_size = [18 if n in top_nodes else 8 for n in sub.nodes()]

fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
                           line=dict(width=0.5, color="lightgrey"), hoverinfo="none"))
fig3.add_trace(go.Scatter(x=node_x, y=node_y, mode="markers",
                           marker=dict(size=node_size, color=node_color, colorscale="Reds",
                                       showscale=True, colorbar=dict(title="risk_score")),
                           text=list(sub.nodes()), hoverinfo="text"))
fig3.update_layout(title="Link-analysis: top-60 flagged transactions + 1-hop neighbours",
                    height=550, width=800, showlegend=False,
                    xaxis=dict(visible=False), yaxis=dict(visible=False))
fig3.show()


In [ ]:
import plotly.io as pio

dashboard_html = "<h1>Bitcoin Transaction Monitoring — Investigative Dashboard</h1>"
for i, fig in enumerate([fig1, fig2, fig3]):
    dashboard_html += pio.to_html(fig, include_plotlyjs=(i == 0), full_html=False)
dashboard_html += "<h2>Top 25 ranked alerts</h2>" + alert_table.to_html(index=False)

with open("dashboard.html", "w") as f:
    f.write(dashboard_html)
print("Saved dashboard.html — open it in a browser for a standalone, offline dashboard.")



## 9. ML

| Focus area | Method | Headline metric |
|---|---|---|
| Entity Clustering | Union-Find (common-input) + TruncatedSVD graph embedding + K-Means | ARI/NMI reported honestly; low ARI traced to a documented dataset property (Section 2), not a code defect |
| Anomaly Detection | Isolation Forest on engineered features | ROC-AUC ≈ 0.76 vs `is_anomaly` |
| Peeling-Chain/Mixing | RandomForest on tx + graph-structural features, plus an explicit chain-walk sequence detector | `mixing` F1 ≈ 1.00, `peeling_chain` recovered further by the sequence detector |
| Risk Scoring | Personalized PageRank from 46 seed wallets over the tx money-flow graph | Positive lift over base rate at wallet level |
| Fusion | Weighted ensemble of the three scorable signals + rule-based reason strings | Ranked, explainable `alerts_top25.csv` |
